# XX - Trade vulnerabilities

Calcul et validation des **métriques de vulnérabilité d'import** (module
`macroforecast.trade.vulnerabilities`) sur les données de commerce international
Eurostat Comext (`DS-045409`) téléchargées par le notebook
*XX - Test orchestration eurostat*.

Trois métriques (documentation, sections 1.1.1-1.1.3), calculées par
`date x nomenclature x indicateur x flux x pays`, à partir du lien entre le pays
et ses partenaires :

- **HHI** = somme des sᵢ² — concentration géographique des sources
  (sᵢ = valeurᵢ / WORLD, sur les pays individuels). Calculé pour tous les flux.
- **CDI2** = imports extra-UE (`EXT_EU`) / imports totaux (`WORLD`) — dépendance
  extra-régionale. Flux d'import uniquement.
- **CDI3** = imports extra-UE (`EXT_EU`, import) / exports totaux (`WORLD`, export)
  — substituabilité domestique. Flux d'import uniquement.


## Importation des modules

In [1]:
# Importation des modules
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# Racine du dépôt ajoutée au path pour importer le package depuis les sources
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from macroforecast.trade.vulnerabilities import run_vulnerabilities, DEFAULT_CONFIG

# Emplacements des catalogues DuckLake (source téléchargée et résultat)
SOURCE_CATALOG = ROOT / "data" / "eurostat.ducklake"
SOURCE_DATA = ROOT / "data" / "eurostat"
SOURCE_SCHEMA = "DS_045409"
RESULT_CATALOG = ROOT / "data" / "trade_vulnerabilities.ducklake"
RESULT_DATA = ROOT / "data" / "trade_vulnerabilities"
RESULT_SCHEMA = "vulnerabilities"


In [ ]:
def _attach(
    catalog: Path, data: Path, alias: str, *, read_only: bool
) -> duckdb.DuckDBPyConnection:
    """Ouvre une connexion DuckDB attachée à un catalogue DuckLake fichier.

    OVERRIDE_DATA_PATH tolère un chemin de données normalisé différemment
    (casse de la lettre de lecteur, séparateurs) de celui stocké dans le catalogue :
    indispensable sous Windows/OneDrive, et non exposé par ``DuckLakeConnector``,
    d'où cet ``ATTACH`` monté à la main.

    Args:
        catalog: Chemin du fichier catalogue ``.ducklake``.
        data: Répertoire des fichiers Parquet.
        alias: Alias du catalogue attaché.
        read_only: Ouverture en lecture seule.

    Returns:
        Connexion DuckDB attachée.
    """
    conn = duckdb.connect()
    conn.execute("INSTALL ducklake; LOAD ducklake;")
    options = f"DATA_PATH '{data.as_posix()}/', OVERRIDE_DATA_PATH true"
    if read_only:
        options += ", READ_ONLY"
    conn.execute(f"ATTACH 'ducklake:{catalog.as_posix()}' AS {alias} ({options})")
    return conn


def attach_readonly(catalog: Path, data: Path, alias: str) -> duckdb.DuckDBPyConnection:
    """Ouvre une connexion en lecture seule sur un catalogue DuckLake.

    Args:
        catalog: Chemin du fichier catalogue ``.ducklake``.
        data: Répertoire des fichiers Parquet.
        alias: Alias du catalogue attaché.

    Returns:
        Connexion DuckDB attachée, en lecture seule.
    """
    return _attach(catalog, data, alias, read_only=True)


def attach_writable(
    catalog: Path, data: Path, alias: str, schema: str
) -> duckdb.DuckDBPyConnection:
    """Ouvre une connexion en écriture et garantit l'existence du schéma cible.

    Args:
        catalog: Chemin du fichier catalogue ``.ducklake``.
        data: Répertoire des fichiers Parquet (créé si absent).
        alias: Alias du catalogue attaché.
        schema: Schéma à activer, créé s'il n'existe pas encore.

    Returns:
        Connexion DuckDB attachée, en écriture.
    """
    data.mkdir(parents=True, exist_ok=True)
    conn = _attach(catalog, data, alias, read_only=False)
    conn.execute(f"CREATE SCHEMA IF NOT EXISTS {alias}.{schema}")
    return conn

## Aperçu des données source

La table de faits source contient les imports (`flow=1`), exports (`flow=2`) et
ré-exports (`flow=3`), pour la masse (`QUANTITY_IN_100KG`) et la valeur
(`VALUE_IN_EUROS`), avec une ventilation par partenaire (`partner`).

In [3]:
# Aperçu : nombre d'observations et de partenaires par flux et indicateur
conn = attach_readonly(SOURCE_CATALOG, SOURCE_DATA, "s")
overview = conn.execute(f"""
    SELECT flow, indicators, count(*) AS n_obs, count(DISTINCT partner) AS n_partners
    FROM s.{SOURCE_SCHEMA}.fact_table
    GROUP BY flow, indicators
    ORDER BY flow, indicators
""").df()
conn.close()
overview


,flow,indicators,n_obs,n_partners
0,1,QUANTITY_IN_100KG,2880,117
1,1,VALUE_IN_EUROS,1962,117
2,2,QUANTITY_IN_100KG,2149,122
3,2,VALUE_IN_EUROS,2149,122


## Calcul des métriques de vulnérabilité

`run_vulnerabilities` lit la table de faits source, applique l'ensemble des
métriques enregistrées (itérables et extensibles) et écrit le résultat dans un
catalogue DuckLake dédié : une colonne par métrique, indexée par
`freq x reporter x product x flow x indicators x TIME_PERIOD`.

In [ ]:
# Calcul et export vers le catalogue résultat.
# Les connexions appartiennent à l'appelant : le runner ne les ouvre ni ne les ferme.
src = attach_readonly(SOURCE_CATALOG, SOURCE_DATA, "s")
res = attach_writable(RESULT_CATALOG, RESULT_DATA, "r", RESULT_SCHEMA)
try:
    report = run_vulnerabilities(
        src,
        source_catalog_alias="s",
        source_schema=SOURCE_SCHEMA,
        result_schema=RESULT_SCHEMA,
        result_conn=res,
        result_catalog_alias="r",
    )
finally:
    src.close()
    res.close()
report

## Table résultat

Cellules d'import disposant des trois métriques (CDI3 nécessite des exports).

In [5]:
# Extrait de la table résultat (cellules d'import avec les trois métriques)
conn = attach_readonly(RESULT_CATALOG, RESULT_DATA, "r")
result_df = conn.execute(f"""
    SELECT freq, reporter, product, flow, indicators, TIME_PERIOD, HHI, CDI2, CDI3
    FROM r.{RESULT_SCHEMA}.fact_table
    WHERE CDI3 IS NOT NULL
    ORDER BY indicators, product, TIME_PERIOD
    LIMIT 15
""").df()
conn.close()
result_df


,freq,reporter,product,flow,indicators,TIME_PERIOD,HHI,CDI2,CDI3
0,A,AT,1,1,QUANTITY_IN_100KG,1999,0.835983,0.012401,0.011207
1,A,AT,1,1,QUANTITY_IN_100KG,2000,0.940787,0.012961,0.015463
2,A,AT,1,1,QUANTITY_IN_100KG,2001,0.900189,0.013946,0.019618
3,A,AT,1,1,QUANTITY_IN_100KG,2002,0.930130,0.009259,0.018341
4,A,AT,1,1,QUANTITY_IN_100KG,2003,0.939517,0.016698,0.028708
5,A,AT,1,1,QUANTITY_IN_100KG,2004,0.704649,0.023019,0.048639
6,A,AT,1,1,QUANTITY_IN_100KG,2005,0.599611,0.007699,0.017912
7,A,AT,1,1,QUANTITY_IN_100KG,2006,0.467326,0.010385,0.024745
8,A,AT,1,1,QUANTITY_IN_100KG,2007,0.403119,0.000727,0.001615
9,A,AT,1,1,QUANTITY_IN_100KG,2008,0.341251,0.000640,0.001632


## Validation numérique

Pour un échantillon de cellules, on recalcule indépendamment chaque métrique en
SQL pur sur la base source, puis on vérifie l'égalité avec la sortie du runner :

- **HHI** = somme des `(valeurᵢ / WORLD)²` sur les pays individuels (hors agrégats :
  `WORLD`, `QW`, et tout code contenant `_`) ;
- **CDI2** = `EXT_EU / WORLD` (imports) ;
- **CDI3** = `EXT_EU (import) / WORLD (export)`.

In [6]:
# Recalcul SQL indépendant et comparaison avec la sortie du runner
src = attach_readonly(SOURCE_CATALOG, SOURCE_DATA, "s")
res = attach_readonly(RESULT_CATALOG, RESULT_DATA, "r")

sample = res.execute(f"""
    SELECT freq, reporter, product, flow, indicators, TIME_PERIOD, HHI, CDI2, CDI3
    FROM r.{RESULT_SCHEMA}.fact_table
    WHERE CDI3 IS NOT NULL
    ORDER BY indicators, product, TIME_PERIOD
    LIMIT 10
""").df()

checks = []
for _, row in sample.iterrows():
    cell = (
        f"reporter='{row.reporter}' AND product='{row['product']}' "
        f"AND indicators='{row.indicators}' AND TIME_PERIOD='{row.TIME_PERIOD}' "
        f"AND freq='{row.freq}'"
    )
    # HHI : somme des parts carrées des pays individuels (part = valeur / WORLD)
    hhi = src.execute(f"""
        WITH w AS (
            SELECT OBS_VALUE AS world FROM s.{SOURCE_SCHEMA}.fact_table
            WHERE {cell} AND flow=1 AND partner='WORLD'
        ),
        ind AS (
            SELECT OBS_VALUE AS v FROM s.{SOURCE_SCHEMA}.fact_table
            WHERE {cell} AND flow=1 AND partner IS NOT NULL
              AND partner NOT IN ('WORLD', 'QW') AND strpos(partner, '_') = 0
        )
        SELECT sum(power(v / (SELECT world FROM w), 2)) FROM ind
    """).fetchone()[0]
    # CDI2 : imports extra-UE / imports totaux
    cdi2 = src.execute(f"""
        SELECT
          (SELECT OBS_VALUE FROM s.{SOURCE_SCHEMA}.fact_table WHERE {cell} AND flow=1 AND partner='EXT_EU') /
          (SELECT OBS_VALUE FROM s.{SOURCE_SCHEMA}.fact_table WHERE {cell} AND flow=1 AND partner='WORLD')
    """).fetchone()[0]
    # CDI3 : imports extra-UE / exports totaux
    cdi3 = src.execute(f"""
        SELECT
          (SELECT OBS_VALUE FROM s.{SOURCE_SCHEMA}.fact_table WHERE {cell} AND flow=1 AND partner='EXT_EU') /
          (SELECT OBS_VALUE FROM s.{SOURCE_SCHEMA}.fact_table WHERE {cell} AND flow=2 AND partner='WORLD')
    """).fetchone()[0]
    checks.append({
        "cell": f"{row.reporter}/{row['product']}/{row.indicators}/{row.TIME_PERIOD}",
        "HHI_runner": row.HHI, "HHI_sql": hhi,
        "CDI2_runner": row.CDI2, "CDI2_sql": cdi2,
        "CDI3_runner": row.CDI3, "CDI3_sql": cdi3,
    })

src.close()
res.close()
check_df = pd.DataFrame(checks)

# Assertions d'égalité (tolérance flottante)
for name in ["HHI", "CDI2", "CDI3"]:
    assert np.allclose(check_df[f"{name}_runner"], check_df[f"{name}_sql"], rtol=0, atol=1e-9), name
print("Validation réussie : les scores du runner correspondent au recalcul SQL indépendant.")
check_df


Validation réussie : les scores du runner correspondent au recalcul SQL indépendant.


,cell,HHI_runner,HHI_sql,CDI2_runner,CDI2_sql,CDI3_runner,CDI3_sql
0,AT/1/QUANTITY_IN_100KG/1999,0.835983,0.835983,0.012401,0.012401,0.011207,0.011207
1,AT/1/QUANTITY_IN_100KG/2000,0.940787,0.940787,0.012961,0.012961,0.015463,0.015463
2,AT/1/QUANTITY_IN_100KG/2001,0.900189,0.900189,0.013946,0.013946,0.019618,0.019618
3,AT/1/QUANTITY_IN_100KG/2002,0.930130,0.930130,0.009259,0.009259,0.018341,0.018341
4,AT/1/QUANTITY_IN_100KG/2003,0.939517,0.939517,0.016698,0.016698,0.028708,0.028708
5,AT/1/QUANTITY_IN_100KG/2004,0.704649,0.704649,0.023019,0.023019,0.048639,0.048639
6,AT/1/QUANTITY_IN_100KG/2005,0.599611,0.599611,0.007699,0.007699,0.017912,0.017912
7,AT/1/QUANTITY_IN_100KG/2006,0.467326,0.467326,0.010385,0.010385,0.024745,0.024745
8,AT/1/QUANTITY_IN_100KG/2007,0.403119,0.403119,0.000727,0.000727,0.001615,0.001615
9,AT/1/QUANTITY_IN_100KG/2008,0.341251,0.341251,0.000640,0.000640,0.001632,0.001632
